In [115]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [116]:
DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
METHOD = "mhcnuggets"

In [117]:
from data_reading import read_gbm122_data

per_region_total_df , per_region_per_allele_mt_df, per_region_per_allele_wt_df = read_gbm122_data(DATA_PATH)


Region 1, unique (Transcrpit, Epitope pairs): 13197
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*35:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*40:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*03:04: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*04:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
Region 2, unique (Transcrpit, Epitope pairs): 13273
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 5127
allele HLA-B*35:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 51

/home/denklewer/Tools/anacoda3/envs/mhcnuggets/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3361: DtypeWarning: Columns (10) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


Region 3, unique (Transcrpit, Epitope pairs): 15170
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*35:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*40:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*03:04: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*04:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
Region 4, unique (Transcrpit, Epitope pairs): 13757
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 4932
allele HLA-B*35:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 49

# 0. Preparations

In [118]:
sample_region_dict = next(iter(per_region_per_allele_wt_df.values()))
sample_allele_df =  next(iter(sample_region_dict.values()))

ALLELES = list(sample_region_dict.keys())
REGIONS = list(per_region_total_df.keys())
ALLELES, REGIONS

(['HLA-A*03:01', 'HLA-B*35:01', 'HLA-B*40:01', 'HLA-C*03:04', 'HLA-C*04:01'],
 [1, 2, 3, 4, 5])

## 0.1. make result dicts

In [119]:
per_region_per_method_per_allele_wt_predictions = dict()
per_region_per_method_per_allele_mt_predictions = dict()
for region in per_region_total_df.keys():
    per_region_per_method_per_allele_wt_predictions[region] = dict()
    per_region_per_method_per_allele_mt_predictions[region] = dict()

## 0.2 Read decoy data

In [120]:
DECOY_DATA_PATH = "/home/denklewer/Data/decoy/sampled_decoy_data.csv"
decoy_df =  pd.read_csv(DECOY_DATA_PATH)
decoy_df['length'] = decoy_df['peptide'].str.len()

per_allele_decoy = dict()
for allele_name in ALLELES:
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcnuggets

In [121]:
# importing the predict module
from mhcnuggets.src.predict import predict
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

In [122]:
def defineClass(allele_name):
    return 'I'

            
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
    

In [123]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)
for region in REGIONS:
    print(f"region {region}")
    print(f"\t mt")
    filterByLength(per_region_per_allele_mt_df[region])
    print(f"\t wt")
    filterByLength(per_region_per_allele_wt_df[region])

decoy 
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
region 1
	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
region 2
	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
region 3
	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
region 4
	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 

In [124]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "")
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA-A*03:01': 'HLA-A03:01',
 'HLA-B*35:01': 'HLA-B35:01',
 'HLA-B*40:01': 'HLA-B40:01',
 'HLA-C*03:04': 'HLA-C03:04',
 'HLA-C*04:01': 'HLA-C04:01'}

In [126]:
# predicting new line separated peptides present in the peptides_path file 
# for MHC class_I allele HLA-A02:01. Since a MS_BA blend model is available,
# this prediction will use the blended model (HLA-A02:01_BA_to_HLAp.h5)
def predict_with_mhcnuggets(per_allele_df, per_allele_scores_for_pecentiles = None):
    # save to temp dir
    TEMPDIR = '../TMP/'
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        df = df[df['peptide'].notnull()]['peptide'] # remove nans
        df.to_csv(f"{TEMPDIR}mchnuggets_data_{allele_name}", index=False, header=False)    
    # predict
    for allele_name in ALLELES:
        class_to_use = defineClass(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        predict(class_=f'{class_to_use}',
                peptides_path=f'{TEMPDIR}mchnuggets_data_{allele_name}', 
                mhc=target_allele,
                output=f'{TEMPDIR}mchnuggets_predictions_{allele_name}')
    # read predictions from temp
    per_allele_predictions= {
        allele_name :pd.read_csv(f"{TEMPDIR}mchnuggets_predictions_{allele_name}").rename(columns={'ic50':'affinity'}) for allele_name in ALLELES
    }
    
    
    for allele_name in ALLELES:
        source_df = per_allele_df[allele_name].copy()
        source_df = source_df[source_df['peptide'].notnull()]['peptide'] # remove nans
        assert len(source_df) == len(per_allele_predictions[allele_name])
        
        
    for allele_name in per_allele_predictions.keys(): 
        df_result = per_allele_predictions[allele_name]
        # add binder label
        per_allele_predictions[allele_name] = per_allele_df[allele_name].merge(df_result, on=['peptide'], how='left')
        # log-scale affinity
        
        per_allele_predictions[allele_name]['score'] = 1 - np.log(per_allele_predictions[allele_name]['affinity'])/np.log(50000)
        
        if per_allele_scores_for_pecentiles:
            
            # get scores for percentile
            ref_scores = per_allele_scores_for_pecentiles[allele_name].copy()
            ref_scores = ref_scores.sort_values('score')['score']
            pct_increment = 1./len(ref_scores)
            
            scores = per_allele_predictions[allele_name]['score']
#             per_allele_predictions[allele_name]['percentile_rank'] = 100 - (1 +  per_allele_predictions[allele_name]['score'].searchsorted(scores['score']))*pct_increment
#             (0.5 + 0.5*df_train.Values.searchsorted(df_validation.Values, side='left') + 0.5*df_train.Values.searchsorted(df_validation.Values, side='right'))*pct_increment
            
            per_allele_predictions[allele_name]['percentile_rank'] = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
            per_allele_predictions[allele_name]['percentile_rank'] = 100 - per_allele_predictions[allele_name]['percentile_rank']*100
            
    
    
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



## First calculate scores for decoy

In [127]:
per_allele_decoy_predictions = predict_with_mhcnuggets(per_allele_decoy)

---HLA-A03:01
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A03:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
---HLA-B35:01
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B35:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
---HLA-B40:01
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B40:01
No BA_to_HLAp model found, predicting with BA model...
---HLA-C03:04
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-C03:04
No BA_to_HLAp model found, predicting with BA model...
---HLA-C04:01
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-C04:01
No BA_to_HLAp model found, predicting with BA

In [128]:
%%time
for region in REGIONS:
    print(f"-region {region}")
    print(f"--mt")
    per_allele_mt_predictions = predict_with_mhcnuggets(per_region_per_allele_mt_df[region], per_allele_decoy_predictions)
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions  

    print(f"--wt")
    per_allele_wt_predictions = predict_with_mhcnuggets(per_region_per_allele_wt_df[region], per_allele_decoy_predictions)
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions  


-region 1
--mt
---HLA-A03:01
Predicting for 5409 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A03:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
---HLA-B35:01
Predicting for 5409 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B35:01
BA_to_HLAp model found, predicting with BA_to_HLAp model...
---HLA-B40:01
Predicting for 5409 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B40:01
No BA_to_HLAp model found, predicting with BA model...
---HLA-C03:04
Predicting for 5409 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-C03:04
No BA_to_HLAp model found, predicting with BA model...
---HLA-C04:01
Predicting for 5409 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-C04:01
No BA_to_HLAp model found, predicting wi

In [129]:
per_region_per_method_per_allele_wt_predictions[region][METHOD][ALLELES[0]]

,allele,length,peptide,source_sequence_name,affinity,score,percentile_rank
0,HLA-A*03:01,8,MYSDKRNK,seq0,5632.35,0.201806,41.09025
1,HLA-A*03:01,8,GAKNFALK,seq1,5586.94,0.202554,6.64550
2,HLA-A*03:01,8,AFYVGGSK,seq5,5623.59,0.201950,27.85225
3,HLA-A*03:01,8,TPPRTRNK,seq6,5620.86,0.201995,24.58250
4,HLA-A*03:01,8,AVCRPLHY,seq9,4742.66,0.217696,0.74425
...,...,...,...,...,...,...,...
4942,HLA-A*03:01,11,QTQPQPQPQPA,seq59723,5688.79,0.200884,95.92225
4943,HLA-A*03:01,11,EPGEPISNHTP,seq59727,5643.62,0.201621,60.74000
4944,HLA-A*03:01,11,PQPQPQPQPEP,seq59738,5639.31,0.201692,53.34975
4945,HLA-A*03:01,11,LKEKMNQVEDE,seq59741,5632.14,0.201809,40.75575


# 2. Join predictions

In [130]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df
    

def getPredictionResult(total_results, region, allele_name, method):
    df = total_results[region][method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df


In [131]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_region_per_allele_result = dict()
    for region in REGIONS:
        print(f"region {region}")
        per_region_per_allele_result[region] = dict()
        methods = list(per_region_per_method_per_allele_mt_predictions[region].keys())
        for allele_name in ALLELES:
            print(f"allele {allele_name}")
            df_source_wt = per_region_per_allele_wt_df[region][allele_name].copy()
            df_source_mt = per_region_per_allele_mt_df[region][allele_name].copy()
            
            for method in methods:
                print(f"method {method}")
                df_method_wt = getPredictionResult(per_region_per_method_per_allele_wt_predictions, region, allele_name, method)
                df_method_mt = getPredictionResult(per_region_per_method_per_allele_mt_predictions, region, allele_name, method)
                df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
                df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
            leave_columns = [ 'source_sequence_name', 'allele', 'length']
            df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
            df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
            # rename id back 

            
            df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
            per_region_per_allele_result[region][allele_name] = df_result
    return per_region_per_allele_result
                
                

              

In [132]:
per_region_per_allele_result = joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions)

region 1
allele HLA-A*03:01
method mhcnuggets
allele HLA-B*35:01
method mhcnuggets
allele HLA-B*40:01
method mhcnuggets
allele HLA-C*03:04
method mhcnuggets
allele HLA-C*04:01
method mhcnuggets
region 2
allele HLA-A*03:01
method mhcnuggets
allele HLA-B*35:01
method mhcnuggets
allele HLA-B*40:01
method mhcnuggets
allele HLA-C*03:04
method mhcnuggets
allele HLA-C*04:01
method mhcnuggets
region 3
allele HLA-A*03:01
method mhcnuggets
allele HLA-B*35:01
method mhcnuggets
allele HLA-B*40:01
method mhcnuggets
allele HLA-C*03:04
method mhcnuggets
allele HLA-C*04:01
method mhcnuggets
region 4
allele HLA-A*03:01
method mhcnuggets
allele HLA-B*35:01
method mhcnuggets
allele HLA-B*40:01
method mhcnuggets
allele HLA-C*03:04
method mhcnuggets
allele HLA-C*04:01
method mhcnuggets
region 5
allele HLA-A*03:01
method mhcnuggets
allele HLA-B*35:01
method mhcnuggets
allele HLA-B*40:01
method mhcnuggets
allele HLA-C*03:04
method mhcnuggets
allele HLA-C*04:01
method mhcnuggets


# 5. For now just write to files

In [133]:
DATA_PATH = "/home/denklewer/Data/gbm122/"

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [134]:
# for decoy_key in DECOY_PREFIXES:

for region in REGIONS:
    for allele_name in ALLELES:
        total_df = per_region_per_allele_result[region][allele_name]
        allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        total_df.to_csv("{}{}_predictions_reg_{}_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,region,allele_name),sep=";", index=False)

In [135]:
total_df

,allele,length,peptide_mt,source_sequence_name,mhcnuggets_score_mt,mhcnuggets_affinity_mt,mhcnuggets_percentile_rank_mt,peptide_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,mhcnuggets_percentile_rank_wt
0,HLA-C*04:01,8,TPRTRNKF,seq14936,0.149838,9882.88,58.94875,PPRTRNKF,0.154883,9357.90,47.45175
1,HLA-C*04:01,8,SWPGAPVY,seq14939,0.181599,7008.77,13.47775,SWPGTPVY,0.187361,6585.16,10.20375
2,HLA-C*04:01,8,YVAVCHPL,seq14940,0.200572,5708.08,5.40475,YVAVCRPL,0.187132,6601.53,10.29925
3,HLA-C*04:01,8,RELLSLAF,seq14941,0.237309,3835.85,1.02575,RGLLSLAF,0.222611,4497.06,1.97675
4,HLA-C*04:01,8,NQYASVAL,seq14942,0.177848,7299.08,16.15675,NaN,NaN,NaN,-0.00025
...,...,...,...,...,...,...,...,...,...,...,...
4942,HLA-C*04:01,11,LHGTLQKDGWT,seq65802,0.137834,11253.59,89.18675,LRGTLQKDGWT,0.136126,11463.46,92.56975
4943,HLA-C*04:01,11,SSSSSSSNNNY,seq65806,0.129699,12289.03,99.39225,SSSSSSNNNNY,0.127711,12556.30,99.88775
4944,HLA-C*04:01,11,RSSSSSSSSSN,seq65807,0.139098,11100.75,86.35125,RSSSSSSSSNN,0.138174,11212.35,88.44675
4945,HLA-C*04:01,11,SSSSSSSSSNN,seq65808,0.141575,10807.19,80.22425,SSSSSSSSNNN,0.139292,11077.52,85.92775
